# Chunker

In [1]:
## Librerías

import os
from typing import List
import torch
from pymongo import MongoClient
from transformers import AutoModel, AutoTokenizer
import docx
import PyPDF2
import datetime

In [2]:
# Funciones

def extract_text(file_path: str) -> str:
    """Extract plain text from a PDF or DOCX document."""
    if file_path.lower().endswith(".pdf"):
        reader = PyPDF2.PdfReader(file_path)
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    if file_path.lower().endswith(".docx"):
        document = docx.Document(file_path)
        return "\n".join(p.text for p in document.paragraphs)
    raise ValueError(f"Unsupported file type: {file_path}")


def chunk_text(text: str, *, chunk_size: int = 200, overlap: int = 20) -> List[str]:
    """Split text into chunks of approximately ``chunk_size`` words."""
    words = text.split()
    step = max(1, chunk_size - overlap)
    return [
        " ".join(words[i : i + chunk_size])
        for i in range(0, len(words), step)
        if words[i : i + chunk_size]
    ]


def embed_text(model, tokenizer, chunk: str):
    """Create a vector embedding for a text chunk."""
    inputs = tokenizer(chunk, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    # Mean pooling over the token dimension produces a single vector
    return outputs.last_hidden_state.mean(dim=1).squeeze().tolist()


def store_chunks(base_dir: str, mongo_uri: str, db_name: str = "rag", collection: str = "documents") -> None:
    """Traverse ``base_dir`` and store document chunks in MongoDB.

    Parameters
    ----------
    base_dir:
        Root directory to search for documents.
    mongo_uri:
        Connection string for MongoDB.
    db_name:
        Target database name. Defaults to ``rag``.
    collection:
        Target collection name. Defaults to ``documents``.
    """
    client = MongoClient(mongo_uri)
    coll = client[db_name][collection]
    print(f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Cargando modelo db y coll cargados {coll}")
    model_name = os.environ.get('modelhugface')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    print(f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Modelo cargado")

    for root, _, files in os.walk(base_dir):
        topic = os.path.basename(root)
        for fname in files:
            if not fname.lower().endswith((".pdf", ".docx")):
                continue
            path = os.path.join(root, fname)
            try:
                print(f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {fname} extrayendo")
                text = extract_text(path)
                print(f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {fname} extraido")
            except:
                pass
            chunks = 0
            for chunk in chunk_text(text):
                print(chunk)
                embedding = embed_text(model, tokenizer, chunk)
                coll.insert_one(
                    {
                        "topic": topic,
                        "filename": fname,
                        "text": chunk,
                        "embedding": embedding,
                    }
                )
                chunks = chunk
            print(f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {fname} completado, {chunks} procesados")
    client.close()

In [3]:
#Definimos variables
mongo_uri = os.environ.get('cs_mongo')
folder = os.environ.get('pathdoc')
db = os.environ.get('mongo')
collection = os.environ.get('col')

In [4]:
archivos = []
for root, _, files in os.walk(folder):
    topic = os.path.basename(root)
    for fname in files:
        if not fname.lower().endswith((".pdf", ".docx")):
            continue
        path = os.path.join(root, fname)
        archivos.append(path)
archivos[0]

'C:\\Users\\alejandro.casares\\Documents\\2 - Industria\\RAG_POC\\data\\docs\\Economia\\Adam Smith - La riqueza de las naciones.pdf'

In [5]:
store_chunks(folder,
              mongo_uri,
              db,
            collection)

[2026-02-11 18:20:37] Cargando modelo db y coll cargados Collection(Database(MongoClient(host=['ac-x54ssck-shard-00-02.rid2jag.mongodb.net:27017', 'ac-x54ssck-shard-00-01.rid2jag.mongodb.net:27017', 'ac-x54ssck-shard-00-00.rid2jag.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, appname='Cluster0RAG1', authsource='admin', replicaset='atlas-ivqk0b-shard-0', tls=True), 'rag'), 'documents')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[2026-02-11 18:20:39] Modelo cargado
[2026-02-11 18:20:39] Adam Smith - La riqueza de las naciones.pdf extrayendo


unknown widths : 
[0, IndirectObject(8543, 0, 1999093610352)]
unknown widths : 
[0, IndirectObject(8538, 0, 1999093610352)]
unknown widths : 
[0, IndirectObject(8533, 0, 1999093610352)]
unknown widths : 
[0, IndirectObject(8523, 0, 1999093610352)]
unknown widths : 
[0, IndirectObject(8518, 0, 1999093610352)]
unknown widths : 
[0, IndirectObject(8513, 0, 1999093610352)]
unknown widths : 
[0, IndirectObject(8503, 0, 1999093610352)]
unknown widths : 
[0, IndirectObject(5712, 0, 1999093610352)]
unknown widths : 
[0, IndirectObject(5757, 0, 1999093610352)]
unknown widths : 
[0, IndirectObject(5757, 0, 1999093610352)]
unknown widths : 
[0, IndirectObject(5712, 0, 1999093610352)]
unknown widths : 
[0, IndirectObject(5850, 0, 1999093610352)]
unknown widths : 
[0, IndirectObject(5757, 0, 1999093610352)]
unknown widths : 
[0, IndirectObject(5712, 0, 1999093610352)]
unknown widths : 
[0, IndirectObject(5712, 0, 1999093610352)]
unknown widths : 
[0, IndirectObject(5757, 0, 1999093610352)]
unknown 

[2026-02-11 18:21:04] Adam Smith - La riqueza de las naciones.pdf extraido
1 1 1 t i ' Edición de Carlos Rodríguez Braun .._ ---=-�� -·- --··--1 i l ¡ 1 1 1' ALIANZA EDITORIAL AdamSmith: La riqueza de las naciones (Libros I-II-III y selección de los Libros IV y V) Estudio preliminar: Carlos Rodríguez Braun El Libro de Bolsillo Alianza Editorial Madrid ® Título original: An Inquiry into the Nature and Causes of the Wealth of Nations. Obra publicada originalmente en dos volúmenes en Londres en 1776. Traductor: Carlos Rodríguez Braun Primera edición en «El Libro de Bolsillo»: 1994 Primera reimpresión en «El Libro de Bolsillo»: 1996 Reservados todos los derechos. De conformidad con lo dispuesto en el art. 534-bis del Código Penal vigente, podrán ser castigados con penas de multa y privación de libertad quienes reprodujeren o plagia· ren, en todo o en parte, una obra literaria, artística o científica fijada en cualquier tipo de soporte sin la preceptiva autorización. © De la traducción y es

In [6]:
from pymongo import MongoClient

client = MongoClient(mongo_uri)
coll = client[db][collection]

print("Número de documentos insertados:", coll.count_documents({}))
# print("Primer documento:", coll.find_one())


Número de documentos insertados: 2126


In [8]:
#Verificamos que haya funcionado correctamente:
doc = coll.find_one()
len(doc["embedding"])


384